In [ ]:
## Task 4 — NOVA Brand Voice Fine-Tuning

This notebook fine-tunes a small instruct model to produce customer support responses in NOVA’s brand voice using:
- supervised fine-tuning
- PEFT LoRA / QLoRA-style training
- 4-bit quantization for memory efficiency
- W&B tracking for experiment logging

Goal:
Transform plain support responses into a warmer, more premium, more reassuring NOVA-style tone.

In [1]:
!pip -q install transformers datasets peft trl accelerate bitsandbytes



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer


c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# W&B is optional. If you don't have an account, just skip logging.
os.environ["WANDB_DISABLED"] = "true"


In [9]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
import os

BASE_DIR = os.getcwd()  # should be C:\Users\273441\Downloads\AIPlatform
TRAIN_FILE = os.path.join(BASE_DIR,  "brand_voice_train.jsonl")
EVAL_FILE = os.path.join(BASE_DIR,  "brand_voice_eval.jsonl")
OUTPUT_DIR = os.path.join(BASE_DIR,  "task4_outputs")


In [18]:
train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
eval_ds = load_dataset("json", data_files=EVAL_FILE, split="train")

print(train_ds[0])
print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))

{'text': "### Instruction:\nRewrite the support response in NOVA's brand voice.\n\n### Input:\nYour order has been shipped and should arrive by Feb 14.\n\n### Response:\nGreat news — your order is already on the way and should reach you by Feb 14. You’re almost there ✨"}
Train size: 15
Eval size: 5


In [19]:
# Ensure dataset has a single 'text' field for SFTTrainer
def format_example(example):
    # Expected fields in JSONL: 'prompt' and 'response'
    prompt = example.get('prompt', '').strip()
    response = example.get('response', '').strip()
    return {"text": f"{prompt}\n{response}"}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)
print(train_ds[0])


Map: 100%|██████████| 5/5 [00:00<00:00, 245.73 examples/s]

{'text': '\n'}


In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [21]:
use_4bit = torch.cuda.is_available()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [22]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if use_4bit else None,
    device_map="auto" if use_4bit else None
)

if not use_4bit:
    model.to("cpu")

model.config.use_cache = False

# Avoid max_length vs max_new_tokens warning
model.generation_config.max_length = None


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2101.12it/s]


In [23]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

In [24]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_steps=10,
    save_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    run_name="nova-brand-voice-qlora",
    fp16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"
)


In [25]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    args=training_args
)

Truncating eval dataset: 100%|██████████| 5/5 [00:00<00:00, 243.00 examples/s]


In [26]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
6,2.222157,1.309678


TrainOutput(global_step=6, training_loss=2.083898882071177, metrics={'train_runtime': 76.7394, 'train_samples_per_second': 0.586, 'train_steps_per_second': 0.078, 'total_flos': 1119706398720.0, 'train_loss': 2.083898882071177})

In [27]:
trainer.model.save_pretrained(os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))

print("Saved adapter to:", os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))

Saved adapter to: c:\Users\273441\Downloads\AIPlatform\task4\task4_outputs\nova_brand_voice_adapter


In [31]:
def generate_response(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Return only the generated continuation (not the prompt)
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


In [32]:
eval_samples = [
    "Rewrite in NOVA brand voice: Your order has been shipped and will arrive by Feb 14.",
    "Rewrite in NOVA brand voice: Please share your order ID.",
    "Rewrite in NOVA brand voice: This issue will be escalated to a human agent.",
    "Rewrite in NOVA brand voice: This product is suitable for dry skin."
]

for s in eval_samples:
    print("\n" + "=" * 100)
    print("PROMPT:", s)
    print("OUTPUT:")
    print(generate_response(s, max_new_tokens=80))

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: Rewrite in NOVA brand voice: Your order has been shipped and will arrive by Feb 14.
OUTPUT:


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Use a call-to-action (CTA) to encourage customers to take action: Click here to place your order now and get your new product delivered by Feb 14.

3. Use a clear and concise message: Shop now and get your new product delivered by Feb 14.

4. Include a discount code or

PROMPT: Rewrite in NOVA brand voice: Please share your order ID.
OUTPUT:


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Email:
Subject: Order #12345

Dear [Customer Name],

We are delighted to inform you that your order #12345 has been successfully placed. We appreciate your business and look forward to serving you in the future.

If you have any questions or concerns, please do not hesitate to contact us at

PROMPT: Rewrite in NOVA brand voice: This issue will be escalated to a human agent.
OUTPUT:


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

PROMPT: Rewrite in NOVA brand voice: This product is suitable for dry skin.
OUTPUT:
2. "Nova Skin Care: The Ultimate Solution for Dry Skin"

3. "Nova Skin Care: The Ultimate Solution for Dry Skin"

4. "Nova Skin Care: The Ultimate Solution for Dry Skin"

5. "Nova Skin Care: The Ultimate Solution for D


In [35]:
results = []

for s in eval_samples:
    out = generate_response(s, max_new_tokens=None)
    results.append({
        "prompt": s,
        "output": out
    })

with open(os.path.join(OUTPUT_DIR, "task4_eval_outputs.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved evaluation outputs.")

KeyboardInterrupt: 